# Bayesian Optimization for Black-Box Functions (Submission 13)

This notebook implements  **"The Final Harvest"** Bayesian Optimization pipeline configured for **Round 13 Queries** across all 8 independent black-box target functions.

### Core Strategic Configurations for Round 13:
* **Maximum Global Search Resolution (300 Multi-Starts):** To completely guard against multi-modal stagnation and find global maxima, the acquisition function surface is scanned with **300 random multi-starts** per function via L-BFGS-B optimization steps.
* **"The Final Harvest" $\xi$-Mapping Architecture:**
  * **$\xi = 0.5000$ (Max Boundary Exploration):** Maintained across Functions 1, 3, 4, and 6 to continue driving wide discovery, leveraging the massive positive momentum discovered in Function 4 and bursting through flat or zero-value dead-zones.
  * **$\xi = 0.1000$ (Noisy Landscape Regularization):** Kept on **Function 2** to systematically regularize broad exploration against stochastic observation variance.
  * **$\xi = 0.0001$ (Hyper-Refinement Convergence):** Deployed for **Function 5**. Moving decisively away from the recent recovery step, this narrows the window to hyper-refine and lock down the massive **1565.23 peak summit**.
  * **$\xi = 0.0000$ (Absolute Pure Exploitation):** Dedicated strictly to **Function 7** and **Function 8** to track immediate local gradients, stabilizing the **2.67 peak** on Function 7 and approaching the maximum convergence limit on the **10.0 target** for Function 8.

In [ ]:
import numpy as np
import os
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel
from scipy.optimize import minimize
from scipy.stats import norm

# Enforce notebook inline rendering support
%matplotlib inline 

print("Environment initialized. Round 13 Final Harvest pipeline active.")

## Definition of Object-Oriented Bayesian Optimizer

In [ ]:
class BayesianOptimizer:
    def __init__(self, is_noisy=False):
        """
        Initializes the noise variance configuration matrix.
        """
        # alpha=1e-2 preserves noise handling bounds for F2, 1e-6 maintains precision for others
        self.alpha = 1e-2 if is_noisy else 1e-6
        self.model = None
        self.X = None
        self.Y = None
        self.dim = None

    def load_and_fit(self, func_dir):
        """
        Loads cumulative histories and fits a robust Gaussian Process Regressor.
        """
        X = np.load(os.path.join(func_dir, "initial_inputs.npy"))
        Y = np.load(os.path.join(func_dir, "initial_outputs.npy"))
        self.X, self.Y, self.dim = X, Y, X.shape[1]
        
        # Standard Matern 2.5 kernel balances functional smoothness and localized adjustments
        kernel = ConstantKernel(1.0) * Matern(
            length_scale=np.ones(self.dim), 
            nu=2.5
        )
        
        self.model = GaussianProcessRegressor(
            kernel=kernel, 
            alpha=self.alpha,
            normalize_y=True, 
            n_restarts_optimizer=50 
        )
        self.model.fit(self.X, self.Y)

    def expected_improvement(self, x, xi):
        """
        Computes Expected Improvement (EI) surfaces for specified jitter configurations.
        """
        x = x.reshape(-1, self.dim)
        mu, sigma = self.model.predict(x, return_std=True)
        mu_sample_opt = np.max(self.Y)

        with np.errstate(divide='warn'):
            imp = mu - mu_sample_opt - xi
            Z = imp / sigma
            ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
            ei[sigma <= 0.0] = 0.0
        return ei

    def propose_next_point(self, xi):
        """
        Optimizes the acquisition function via 300 random multi-start bounds searches.
        """
        best_x = None
        max_ei = -1
        
        for _ in range(300):
            x0 = np.random.uniform(0.0, 1.0, self.dim)
            res = minimize(
                lambda x: -self.expected_improvement(x, xi),
                x0,
                bounds=[(0.0, 1.0)] * self.dim,
                method='L-BFGS-B'
            )
            if -res.fun > max_ei:
                max_ei = -res.fun
                best_x = res.x
        return best_x

## Sequential Execution Optimization Pipeline

In [ ]:
output_file = "submission_13_results.txt"

# xi_map Logic: "The Final Harvest"
# F1, F3, F4, F6: Maximum discovery jitter (0.5000) to burst forward on momentum.
# F2: Balanced regularized exploration (0.1000) for noisy observations.
# F5: Surgical reduction to 0.0001 to converge and pin down the massive 1565.23 peak.
# F7: Pure exploitation (0.0000) to narrow focus and settle on the 2.67 peak.
# F8: Pure exploitation (0.0000) combined with 300 restarts to approach the 10.0 target.
xi_map = {
    1: 0.5000, 
    2: 0.1000, 
    3: 0.5000, 
    4: 0.5000, 
    5: 0.0001,  # Deep refinement pivot
    6: 0.5000, 
    7: 0.0000,  # Peak lock-in
    8: 0.0000   # Extreme gradient search
}

print("Generating Submission 13: The Final Harvest...")
print("-" * 55)

notebook_results = {}

for i in range(1, 8 + 1):
    func_dir = f"function_{i}"
    
    if not os.path.exists(func_dir):
        print(f"Warning: Functional target directory context '{func_dir}' not found.")
        continue
        
    optimizer = BayesianOptimizer(is_noisy=(i == 2))
    optimizer.load_and_fit(func_dir)
    
    current_xi = xi_map[i]
    next_point = optimizer.propose_next_point(xi=current_xi)
    formatted_point = "-".join([f"{val:.6f}" for val in next_point])
    
    notebook_results[f"Function {i}"] = (formatted_point, current_xi)
    print(f"Function {i}: {formatted_point} (xi={current_xi})")

# Commit finalized coordinates to results text document
with open(output_file, "w") as f:
    for i in range(1, 9):
        key = f"Function {i}"
        if key in notebook_results:
            f.write(f"{key}: {notebook_results[key][0]}\n")

print("-" * 55)
print(f"Submission 13 successfully saved to target output: {output_file}")

### Summary Evaluation Matrix

In [ ]:
print(f"| {'Target Channel':15} | {'Exploration (xi)':18} | {'Proposed Query Coordinates (Round 13)':55} |")
print(f"| {'-'*15} | {'-'*18} | {'-'*55} |")
for func, (coords, xi_val) in notebook_results.items():
    print(f"| {func:15} | {xi_val:<18} | {coords:55} |")